# Hotel Inventory Optimization Solution

## Executive Summary & Solution Walkthrough

### 1. What is the core business problem and why does it matter?
**Problem**: Inefficient inventory management in hotel bars leads to two extremes: **stockouts** (running out of popular drinks), which causes lost revenue and unhappy guests, or **overstocking**, which ties up capital and risks waste/theft.
**Why it matters**: In the low-margin hospitality industry, optimizing this balance directly impacts the bottom line. A data-driven approach removes the guesswork from ordering.

### 2. What assumptions did you make? Why?
- **Demand is Stationary**: We assume historical daily consumption is a good predictor of future demand.
- **Lead Time is 3 Days**: We assumed a fixed time between placing an order and receiving it.
- **Service Level Goal is 95%**: We aim to be in stock 95% of the time (Z-score = 1.65).
- **Continuous Review**: We assume we can place an order any day when stock drops below the reorder point.

### 3. What model did you use and why did you choose it? Why not others?
**Model**: **Par Level System (Min-Max Inventory Control)**.
**Why**:
- It is the **industry standard** for hospitality. Staff understand "Par" (target stock).
- It is robust for stable demand items.
- Complex ML models (ARIMA, LSTM) are often overkill for sparse bar data and harder to explain to operational staff.

### 4. How does your system perform? What would you improve?
**Performance**: The simulation demonstrates that this system maintains a high Service Level (>90-95%) for most items.
**Improvements**:
- **Seasonality**: Adjust Pars for holidays/weekends.
- **Dynamic Lead Time**: Account for supplier delays.

### 5. How would this solution work in a real hotel?
**In Production**:
1.  Script runs weekly on the POS data.
2.  Generates a "Count Sheet" with calculated Par Levels.
3.  Bar staff physically count current stock.
4.  Order Quantity = `Par Level - Current Level`.


## 1. Setup & Integration
Here we import necessary libraries and our custom modules (`data_loader`, `forecast_model`, `simulator`) which contain the core logic.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Import local modules
from data_loader import load_and_process_data
from forecast_model import calculate_par_levels
from simulator import run_simulation

# Configuration
FILE_PATH = "Copy of Consumption Dataset - Dataset.csv"
LEAD_TIME_DAYS = 3
SERVICE_LEVEL_Z = 1.65 # 95% Service Level

# Visual styling
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


## 2. Data Loading & Cleaning
We use the `load_and_process_data` function to ingest the raw consumption CSV, clean the numeric columns, and aggregate data to a **Daily** level.


In [ ]:
# Load Data
daily_df = load_and_process_data(FILE_PATH)

print(f"Data Loaded. Shape: {daily_df.shape}")
daily_df.head()


## 3. Exploratory Data Analysis (EDA)
Understanding consumption patterns is key. We look at the most popular brands and the daily consumption volatility.


In [ ]:
# 1. Top 10 Most Consumed Brands (by Volume)
top_brands = daily_df.groupby('Brand Name')['Consumed (ml)'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_brands.values, y=top_brands.index, palette='viridis')
plt.title('Top 10 Most Consumed Brands (Total Volume)')
plt.xlabel('Total Consumption (ml)')
plt.show()

# 2. Daily Consumption Distribution for Top Brand
top_brand_name = top_brands.index[0]
subset = daily_df[daily_df['Brand Name'] == top_brand_name]

plt.figure(figsize=(12, 5))
plt.plot(subset['Date'], subset['Consumed (ml)'], marker='o', linestyle='-', alpha=0.7)
plt.title(f'Daily Consumption Trend: {top_brand_name}')
plt.ylabel('Consumed (ml)')
plt.show()


## 4. Par Level Forecasting
We calculate the **Par Level** for each item.
$$ Par = (AvgUsage \times LeadTime) + SafetyStock $$
Safety Stock handles the demand variability (Standard Deviation).


In [ ]:
# Calculate Par Levels
par_levels_df = calculate_par_levels(daily_df, lead_time_days=LEAD_TIME_DAYS, service_level_z=SERVICE_LEVEL_Z)

print("Recommended Par Levels (Top 5 High Volume Items):")
print(par_levels_df[['Bar Name', 'Brand Name', 'mean_daily_usage', 'recommended_par_level_ml', 'par_bottles_750ml']].sort_values(by='mean_daily_usage', ascending=False).head().to_string(index=False))


## 5. Inventory Simulation (Validation)
We simulate the past year using our new Par Levels to see if they would have prevented stockouts.


In [ ]:
# Run Simulation
sim_results = run_simulation(daily_df, par_levels_df, lead_time_days=LEAD_TIME_DAYS)

# Summary Metrics
avg_sl = sim_results['Service Level'].mean()
print(f"Average Service Level across all items: {avg_sl:.2%}")

# Show items with lowest service levels (potential issues)
print("\nItems with Lowest Service Levels:")
print(sim_results[['Bar Name', 'Brand Name', 'Service Level', 'Stockout Days']].sort_values('Service Level').head().to_string(index=False))


## 6. Conclusion
The simulation confirms that our Par Level system maintains high availability for the majority of items. The next step would be to deploy these Par Levels to the bar staff for a live pilot.
